# 07 Adapter | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: niezgodnosc interfejsow
2. 🔌 Adapter obiektowy (kompozycja)
3. 🧬 Adapter klasowy (wielodziedziczenie)
4. 🔮 `__getattr__` jako adapter dynamiczny
5. 🆚 Adapter vs Facade vs Proxy

## 1. 🔹 Problem: niezgodnosc interfejsow

Adapter (znany tez jako Wrapper) to wzorzec strukturalny pozwalajacy
obszarom wspolpracowac mimo niezgodnych interfejsow.

Analogia: adapter podrozny umozliwia podlaczenie europejskiej
wtyczki do gniazda brityjskiego. Ani wtyczka ani gniazdo nie
sq modyfikowane - adapter tlUmaczy miedzy nimi.

Problem bez Adaptera:
- Stary kod ma jedno API
- Nowa biblioteka ma inne API
- Chcemy uzywac nowej biblioteki bez zmiany starego kodu

> 💡 Adapter zmienia INTERFEJS obiektu. Facade upraszcza interfejs.
> Proxy kontroluje dostep. To sa rozne problemy!

In [ ]:
# Stary interfejs: metoda render()
class LegacyRenderer:
    def render(self, data: list) -> str:
        return f'Legacy: {data}'

# Nowa biblioteka: metoda draw()
class ModernRenderer:
    def draw(self, items: list, style: str = 'default') -> str:
        return f'Modern({style}): {items}'

# Klient oczekuje metody render() - nie moze uzywac ModernRenderer
def display_chart(renderer, data: list) -> None:
    print(renderer.render(data))  # oczekuje interfejsu z render()

display_chart(LegacyRenderer(), [1, 2, 3])   # dziala
# display_chart(ModernRenderer(), [1, 2, 3]) # AttributeError: no render()

---

### 🐍 Cwiczenia - problem

1. Sprawdz jaki blad pojawia sie gdy wywolasz `display_chart(ModernRenderer(), [1,2,3])`.
2. Napisz klase `OldSearch` z metoda `find(query: str) -> list` i
   `NewSearch` z metoda `search(term: str, limit: int) -> list`.
   Pokaz ze sa niezgodne.
3. *(Trudniejsze)* Policz ile miejsc w kodzie musisz zmienic
   jesli masz 5 funkcji uzywajacych `render()` i chcesz przejsc
   na `draw()` bez adaptera.

In [ ]:
# Cwiczenie 1: blad bez adaptera
try:
    display_chart(ModernRenderer(), [1, 2, 3])
except AttributeError as e:
    print(f'Blad: {e}')

In [ ]:
# Cwiczenie 2: niezgodne interfejsy
class OldSearch:
    def find(self, query: str) -> list:
        return [f'result:{query}']

class NewSearch:
    def search(self, term: str, limit: int = 10) -> list:
        return [f'new_result:{term}'] * min(limit, 3)

def search_users(searcher, query: str) -> list:
    return searcher.find(query)  # oczekuje interfejsu OldSearch

print(search_users(OldSearch(), 'Alice'))
try:
    print(search_users(NewSearch(), 'Alice'))  # niezgodny!
except AttributeError as e:
    print(f'Blad: {e}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: koszt zmiany bez adaptera
functions_using_render = ['display_chart', 'render_dashboard', 'export_pdf',
                           'show_preview', 'print_report']
print(f'Miejsc do zmiany: {len(functions_using_render)}')
print('Z adapterem: 0 miejsc w kodzie klienta - tylko 1 klasa adaptera')

## 2. 🔹 Adapter obiektowy (kompozycja)

Adapter obiektowy uzywa kompozycji: trzyma referencje do adaptowanego
obiektu i deleguje wywolania do niego po translacji interfejsu.

Struktura:
- Target: interfejs oczekiwany przez klienta
- Adaptee: istniejacy interfejs do zaadaptowania
- Adapter: implementuje Target, trzyma instancje Adaptee

Zalety kompozycji: mozemy adaptowac konkretna instancje,
nie cala klase; mozna adaptowac obiekty roznych klas.

> 💡 Adapter obiektowy jest preferowanym podejsciem w Pythonie
> ze wzgledu na prostosc i zgodnosc z zasada Dependency Inversion.

In [ ]:
# Adapter dla ModernRenderer
class ModernRendererAdapter:
    def __init__(self, renderer: ModernRenderer):
        self._renderer = renderer   # kompozycja

    def render(self, data: list) -> str:  # implementuje stary interfejs
        return self._renderer.draw(data, style='chart')  # deleguje do nowego

# Klient nie zmieniony
display_chart(LegacyRenderer(), [1, 2, 3])
display_chart(ModernRendererAdapter(ModernRenderer()), [1, 2, 3])  # dziala!

# Pelniejszy przyklad: adapter platnosci
class StripePayment:
    def charge_card(self, card_token: str, amount_cents: int) -> dict:
        return {'status': 'succeeded', 'amount': amount_cents, 'currency': 'usd'}

class PayPalPayment:
    def make_payment(self, email: str, amount: float, currency: str) -> bool:
        print(f'PayPal: {amount} {currency} -> {email}')
        return True

class Payment:  # wspolny interfejs klienta
    def pay(self, amount: float) -> bool: ...

class StripeAdapter(Payment):
    def __init__(self, stripe: StripePayment, card_token: str):
        self._stripe = stripe
        self._token = card_token
    def pay(self, amount: float) -> bool:
        cents = int(amount * 100)
        result = self._stripe.charge_card(self._token, cents)
        return result['status'] == 'succeeded'

class PayPalAdapter(Payment):
    def __init__(self, paypal: PayPalPayment, email: str):
        self._paypal = paypal
        self._email = email
    def pay(self, amount: float) -> bool:
        return self._paypal.make_payment(self._email, amount, 'EUR')

def checkout(payment: Payment, amount: float) -> None:
    success = payment.pay(amount)
    print(f'Payment {"OK" if success else "FAILED"}: {amount}')

checkout(StripeAdapter(StripePayment(), 'tok_xxx'), 49.99)
checkout(PayPalAdapter(PayPalPayment(), 'user@paypal.com'), 49.99)

---

### 🐍 Cwiczenia - adapter obiektowy

1. Napisz `SearchAdapter` adaptujacy `NewSearch.search()` do
   interfejsu `OldSearch.find()`. Przetestuj z `search_users`.
2. Stary interfejs: `Logger.log(level, msg)`. Nowy: `CloudLogger.emit(event: dict)`.
   Napisz adapter w obu kierunkach.
3. *(Trudniejsze)* Napisz `SortingAdapter` adaptujacy sortowanie
   po kluczu (key=) do interfejsu z oddzielnym `comparator(a, b)`.

In [ ]:
# Cwiczenie 1: SearchAdapter
class SearchAdapter:
    def __init__(self, new_search: NewSearch):
        ...
    def find(self, query: str) -> list:
        ...

adapter = SearchAdapter(NewSearch())
print(search_users(adapter, 'Bob'))  # dziala z nowym interfejsem

In [ ]:
# Cwiczenie 2: adapter loggera
class OldStyleLogger:
    def log(self, level: str, msg: str) -> None:
        print(f'[{level}] {msg}')

class CloudLogger:
    def emit(self, event: dict) -> None:
        print(f'CLOUD: {event}')

class CloudLoggerAdapter:  # stary interfejs -> nowy silnik
    def __init__(self, cloud: CloudLogger): ...
    def log(self, level: str, msg: str) -> None: ...

def application_log(logger, msg: str) -> None:
    logger.log('INFO', msg)

application_log(OldStyleLogger(), 'App started')
application_log(CloudLoggerAdapter(CloudLogger()), 'App started')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: adapter komparatora
import functools

def sort_with_comparator(items: list, comparator) -> list:
    # stary interfejs: comparator(a, b) -> int (<0, 0, >0)
    return sorted(items, key=functools.cmp_to_key(comparator))

class KeySortAdapter:
    def __init__(self, key_func):
        self._key = key_func
    def compare(self, a, b) -> int:
        # hint: key(a) < key(b) -> -1, == -> 0, > -> 1
        ka, kb = self._key(a), self._key(b)
        return (ka > kb) - (ka < kb)

items = [{'name': 'Charlie', 'age': 30}, {'name': 'Alice', 'age': 25}, {'name': 'Bob', 'age': 35}]
adapter = KeySortAdapter(lambda x: x['age'])
print(sort_with_comparator(items, adapter.compare))

## 3. 🔹 Adapter klasowy (wielodziedziczenie)

Adapter klasowy uzywa wielodziedziczenia: dziedziczy zarówno
po Target (interfejsie klienta) jak i po Adaptee (klasie zrodlowej).

Struktura: `class Adapter(Target, Adaptee):`
- Implementuje metody Target korzystajac z metod Adaptee
- Nie potrzebuje delegacji - metody Adaptee sa dostepne bezposrednio

Wady:
- Wiaze adapter z konkretna klasa Adaptee (nie instancja)
- Moze powodowac problemy z diamond problem w MRO

> 💡 Adapter obiektowy jest zazwyczaj lepszym wyborem. Adapter klasowy
> stosuj tylko gdy koniecznie musisz dziedziczyc z obu klas.

In [ ]:
class JSONSerializer:
    def to_json(self, data: dict) -> str:
        import json
        return json.dumps(data)

class XMLOutput:
    def to_xml(self, data: dict) -> str: ...  # interfejs docelowy

class JSONToXMLAdapter(XMLOutput, JSONSerializer):  # wielodziedziczenie
    def to_xml(self, data: dict) -> str:
        json_str = self.to_json(data)  # metoda z JSONSerializer
        import json
        obj = json.loads(json_str)
        xml_parts = ['<root>']
        for k, v in obj.items():
            xml_parts.append(f'  <{k}>{v}</{k}>')
        xml_parts.append('</root>')
        return '\n'.join(xml_parts)

adapter = JSONToXMLAdapter()
print(adapter.to_xml({'name': 'Alice', 'age': 30, 'city': 'Warsaw'}))
print(type.mro(JSONToXMLAdapter))   # kolejnosc MRO

---

### 🐍 Cwiczenia - adapter klasowy

1. Napisz `CSVToListAdapter(ListOutput, CSVReader)` konwertujacy
   wynik `CSVReader.read()` na listy slownikow przez `to_list()`.
2. Porownaj adapter klasowy i obiektowy dla tego samego problemu.
   Ktore podejscie jest prostrze i dlaczego?
3. *(Trudniejsze)* Sprawdz MRO (Method Resolution Order) dla
   adaptera klasowego uzywajac `ClassName.__mro__`.

In [ ]:
# Cwiczenie 1: CSVToListAdapter
class CSVReader:
    def read_csv(self, text: str) -> list[list[str]]:
        lines = text.strip().split('\n')
        return [line.split(',') for line in lines]

class ListOutput:
    def to_list(self, text: str) -> list[dict]: ...  # interfejs docelowy

class CSVToListAdapter(ListOutput, CSVReader):
    def to_list(self, text: str) -> list[dict]:
        ...

adapter = CSVToListAdapter()
csv = 'name,age,city\nAlice,30,Warsaw\nBob,25,Krakow'
print(adapter.to_list(csv))

In [ ]:
# Cwiczenie 2: porownanie
print('Adapter klasowy:')
print('+ Prostszy kod (brak self._adaptee)')
print('- Wiazanie z konkretna klasa')
print('- Dziedziczenie zamiast kompozycji')
print()
print('Adapter obiektowy (kompozycja):')
print('+ Mozna adaptowac dowolna instancje')
print('+ Zgodnosc z DIP (zaleznosc od abstrakcji)')
print('+ Latwy do testowania z mockiem')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: MRO
print('MRO dla CSVToListAdapter:')
for cls in CSVToListAdapter.__mro__:
    print(f'  {cls.__name__}')

## 4. 🔹 `__getattr__` jako adapter dynamiczny

`__getattr__(self, name)` jest wywolywane tylko gdy normalny
mechanizm wyszukiwania atrybutu nie znajdzie go. Mozna go
uzyc do dynamicznego przekazywania nieznanych atrybutow
do owijanego obiektu.

Zachowanie:
- Python szuka atrybutu normalnie (instancja -> klasa -> bazy)
- Jesli nie znajdzie: wywoluje `__getattr__(name)`
- `__getattr__` moze zwrocic wartosc lub podniesc `AttributeError`

Roznica od `__getattribute__`:
- `__getattribute__` - wywolywany ZAWSZE przy dostepie do atrybutu
- `__getattr__` - wywolywany TYLKO gdy atrybut nie znaleziony

> 💡 Uzywaj `__getattr__` do przekazywania nieznanych wywolan -
> znane metody implementuj jawnie.

In [ ]:
class LegacyService:
    def get_users(self) -> list: return [{'id': 1, 'login': 'alice'}]
    def get_products(self) -> list: return [{'id': 1, 'title': 'Widget'}]
    def get_orders(self) -> list: return [{'id': 101, 'status': 'pending'}]

class ModernServiceAdapter:
    def __init__(self, legacy: LegacyService):
        self._legacy = legacy

    def users(self) -> list:
        raw = self._legacy.get_users()
        return [{'id': u['id'], 'username': u['login']} for u in raw]

    def __getattr__(self, name: str):
        # Dla nieprzetlumaczonych metod: przekaz do legacy
        legacy_method = getattr(self._legacy, f'get_{name}', None)
        if legacy_method:
            return legacy_method
        raise AttributeError(f'{type(self).__name__} has no attribute {name!r}')

adapter = ModernServiceAdapter(LegacyService())

# Przetlumaczona metoda
print(adapter.users())         # nowoczesny format

# Nieprzetlumaczone - przekazane do legacy
print(adapter.products())     # get_products()
print(adapter.orders())       # get_orders()

try:
    adapter.unknown_method()  # AttributeError
except AttributeError as e:
    print(f'Error: {e}')

---

### 🐍 Cwiczenia - `__getattr__`

1. Napisz `TranslatingAdapter(legacy)` z `__getattr__` przeklada
   atrybuty `get_xxx` na `fetch_xxx` i vice versa.
2. Napisz `DebugAdapter(target)` logujacy WSZYSTKIE wywolania
   przez `__getattr__`.
3. *(Trudniejsze)* Napisz `LazyAdapter(factory_func)` inicjalizujacy
   opakowany obiekt tylko przy pierwszym wywolaniu metody
   (uzywajac `__getattr__`).

In [ ]:
# Cwiczenie 1: TranslatingAdapter
class OldAPI:
    def get_user(self) -> dict: return {'user': 'Alice'}
    def get_order(self) -> dict: return {'order': 101}

class TranslatingAdapter:
    def __init__(self, api: OldAPI): self._api = api
    def __getattr__(self, name: str):
        # fetch_xxx -> get_xxx
        if name.startswith('fetch_'):
            old_name = 'get_' + name[6:]
            return getattr(self._api, old_name)
        raise AttributeError(name)

ta = TranslatingAdapter(OldAPI())
print(ta.fetch_user())   # wywola get_user()
print(ta.fetch_order())  # wywola get_order()

In [ ]:
# Cwiczenie 2: DebugAdapter
class DebugAdapter:
    def __init__(self, target): object.__setattr__(self, '_target', target)
    def __getattr__(self, name: str):
        attr = getattr(object.__getattribute__(self, '_target'), name)
        if callable(attr):
            def logged(*args, **kwargs):
                print(f'DEBUG CALL: {name}({args}, {kwargs})')
                result = attr(*args, **kwargs)
                print(f'DEBUG RETURN: {result!r}')
                return result
            return logged
        return attr

class Calculator:
    def add(self, a, b): return a + b
    def mul(self, a, b): return a * b

debug_calc = DebugAdapter(Calculator())
debug_calc.add(3, 4)
debug_calc.mul(5, 6)

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: LazyAdapter
class LazyAdapter:
    def __init__(self, factory):
        object.__setattr__(self, '_factory', factory)
        object.__setattr__(self, '_instance', None)

    def _get_instance(self):
        if object.__getattribute__(self, '_instance') is None:
            factory = object.__getattribute__(self, '_factory')
            object.__setattr__(self, '_instance', factory())
            print('Lazy: initialized!')
        return object.__getattribute__(self, '_instance')

    def __getattr__(self, name: str):
        return getattr(self._get_instance(), name)

class ExpensiveService:
    def __init__(self): print('ExpensiveService: CREATED')
    def query(self, q: str) -> list: return [q]

lazy = LazyAdapter(lambda: ExpensiveService())
print('Adapter created (no init yet)')
print(lazy.query('SELECT 1'))  # inicjalizuje tutaj
print(lazy.query('SELECT 2'))  # juz zainicjalizowany

## 5. 🔹 Adapter vs Facade vs Proxy

Trzy wzorce sa czesto mylone poniewaz wszystkie "owijaja" obiekt:

| Wzorzec | Cel | Co zmienia |
|---|---|---|
| Adapter | Zgodnosc interfejsow | Interfejs |
| Facade | Uproszczenie | Zlozonos, liczba klas |
| Proxy | Kontrola dostepu | Zachowanie |

**Adapter**: niezgodne interfejsy, chcemy uzywac klasy w kontekscie
ktory oczekuje innego API.

**Facade**: wiele klas, chcemy jeden prosty punkt wejscia ukrywajacy
szczegoly implementacji.

**Proxy**: ten sam interfejs, dodajemy kontrole dostepu, lazy loading,
cache, logowanie.

> 💡 Pytaj o zamiar:
> - "Chce uzyc tej klasy, ale ma zly interfejs" -> Adapter
> - "Chce uproscisc ten skomplikowany podsystem" -> Facade
> - "Chce kontrolowac dostep do tego obiektu" -> Proxy

In [ ]:
# Porownanie trzech wzorcow

class DatabaseService:
    def execute(self, sql: str) -> list:
        return [{'id': 1}]

# ADAPTER: zmiana interfejsu
class DatabaseAdapter:
    def __init__(self, db: DatabaseService):
        self._db = db
    def query(self, sql: str) -> list:  # nowy interfejs
        return self._db.execute(sql)    # stary interfejs wewnatrz

# PROXY: ten sam interfejs + kontrola
class DatabaseProxy(DatabaseService):
    def __init__(self, db: DatabaseService):
        self._db = db
    def execute(self, sql: str) -> list:  # ten sam interfejs
        print(f'LOG: {sql}')              # logowanie
        return self._db.execute(sql)

# FACADE: uproszczenie wielu klas
class AppFacade:
    def __init__(self):
        self._db = DatabaseService()
        # wiecej podsystemow...
    def get_users(self) -> list:  # prosty interfejs wysokiego poziomu
        return self._db.execute('SELECT * FROM users')

print('Adapter:', DatabaseAdapter(DatabaseService()).query('SELECT 1'))
print('Proxy:',   DatabaseProxy(DatabaseService()).execute('SELECT 1'))
print('Facade:',  AppFacade().get_users())

---

### 🐍 Cwiczenia - roznice

1. Napisz przyklad dla kazdego wzorca (Adapter, Facade, Proxy)
   uzywajac tego samego `EmailService` jako bazy.
2. Podaj 3 przyklady z biblioteki standardowej lub popularnych
   bibliotek dla kazdego wzorca.
3. *(Trudniejsze)* Czy jeden obiekt moze byc jednoczesnie Adapterem
   i Proxy? Napisz przyklad (zmiana interfejsu + cache).

In [ ]:
# Cwiczenie 1: trzy wzorce na EmailService
class EmailService:
    def send_email(self, to: str, subject: str, body: str) -> bool:
        print(f'Email to {to}: {subject}')
        return True

# Adapter: nowy interfejs notify(user, msg)
class EmailAdapter:
    def __init__(self, svc: EmailService): self._svc = svc
    def notify(self, user: str, msg: str) -> bool:
        return self._svc.send_email(user, 'Notification', msg)

# Proxy: ten sam interfejs + limitowanie czestotliwosci
class RateLimitedEmailProxy(EmailService):
    def __init__(self, svc: EmailService): self._svc = svc; self._count = 0
    def send_email(self, to, subject, body) -> bool:
        self._count += 1
        if self._count > 3: print('Rate limit!'); return False
        return self._svc.send_email(to, subject, body)

# Facade: upraszcza wysylanie roznych typow emaili
class EmailFacade:
    def __init__(self): self._svc = EmailService()
    def welcome(self, user: str) -> bool: return self._svc.send_email(user, 'Welcome!', 'Hi!')
    def invoice(self, user: str, amount: float) -> bool: return self._svc.send_email(user, 'Invoice', f'{amount}')

EmailAdapter(EmailService()).notify('alice@x.com', 'Hello!')
RateLimitedEmailProxy(EmailService()).send_email('a@x.com', 'Hi', 'Text')
EmailFacade().welcome('bob@x.com')

In [ ]:
# Cwiczenie 2: przyklady z biblioteki standardowej
examples = {
    'Adapter': [
        'io.TextIOWrapper - adaptuje binarny strumien do tekstowego',
        'logging.Handler - adaptuje rozne wyjscia do interfejsu loggera',
        'pathlib.Path - adapter OOP dla os.path',
    ],
    'Facade': [
        'smtplib.SMTP - ukrywa szczegoly protokolu SMTP',
        'sqlite3 - fasada dla silnika bazy danych',
        'zipfile.ZipFile - fasada dla operacji na archiwach ZIP',
    ],
    'Proxy': [
        'weakref.proxy - proxy ze slaba referencja',
        'unittest.mock.MagicMock - proxy do testow',
        'functools.lru_cache - proxy cache dla funkcji',
    ],
}
for pattern, ex in examples.items():
    print(f'\n{pattern}:')
    for e in ex: print(f'  - {e}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Adapter + Proxy w jednym
class OldWeatherAPI:
    def get_weather_data(self, city_code: str) -> dict:
        print(f'[API CALL] city_code={city_code}')
        return {'city': city_code, 'temp': 20, 'humidity': 65}

class WeatherAdapterProxy:
    # Adapter: nowy interfejs get_weather(city_name)
    # Proxy: cache wynikow
    CITY_CODES = {'Warsaw': 'WAW', 'Krakow': 'KRK'}

    def __init__(self, api: OldWeatherAPI):
        self._api = api
        self._cache: dict = {}

    def get_weather(self, city_name: str) -> dict:  # nowy interfejs
        if city_name in self._cache:
            print(f'[CACHE] {city_name}')
            return self._cache[city_name]
        city_code = self.CITY_CODES.get(city_name, city_name)
        result = self._api.get_weather_data(city_code)  # stary interfejs
        self._cache[city_name] = result
        return result

wap = WeatherAdapterProxy(OldWeatherAPI())
print(wap.get_weather('Warsaw'))   # API call
print(wap.get_weather('Warsaw'))   # cache
print(wap.get_weather('Krakow'))   # API call